In [27]:
import os
from dotenv import load_dotenv
load_dotenv()
HuggingFaceApi = os.getenv('HF_TOKEN')
groq_api_key = os.getenv("GROQ_API_KEY")

In [28]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine

engine = create_engine("sqlite:///dataframe.db")
db = SQLDatabase(engine=engine)

In [29]:
print(db.run("Select * From dataframe where Age>=60"))

[('Cynthia Guerrero', 90, 'Port Eric'), ('Connie Chase', 66, 'West Courtney'), ('Jeremy Gonzalez', 74, 'Jeromemouth'), ('Kimberly Casey', 72, 'West Andrea'), ('Aaron Harmon', 64, 'West Natalieshire'), ('Angelica Owens', 89, 'Jacksonberg'), ('Daniel Smith', 63, 'Wilsonstad'), ('Jeffrey Riley', 81, 'Matthewfort'), ('Joseph Johnson', 65, 'Bartlettburgh'), ('Brenda Lewis MD', 72, 'Jonmouth'), ('Sophia Miller', 70, 'South Jennifer'), ('Erin Garrett', 81, 'Brooksview'), ('Mary Harding', 63, 'Danielland'), ('Kenneth Caldwell', 82, 'Carsonmouth'), ('Katie Lopez', 80, 'Jerrybury'), ('Caitlyn Vazquez', 73, 'New Brendaburgh'), ('Walter Carter', 89, 'North William'), ('Elijah Torres', 77, 'Aprilborough'), ('Ellen Simmons MD', 64, 'Scottmouth'), ('Andrew Todd', 90, 'Eugeneside'), ('Kristopher Dillon', 79, 'Justinland'), ('Erin Valentine', 81, 'Sanchezburgh'), ('Gregory Boyd', 86, 'Tatemouth'), ('Regina Branch', 88, 'North Tina'), ('Mark Dennis', 65, 'West Leahshire'), ('Maurice Jones', 72, 'East Sa

In [39]:
from langchain_groq import ChatGroq
llm = ChatGroq(model = "llama3-8b-8192",api_key=groq_api_key,
               temperature= 1, max_tokens=1024
)

In [40]:

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

tools = toolkit.get_tools()

tools

[QuerySQLDataBaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001F288A8F0D0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001F288A8F0D0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001F288A8F0D0>),
 QuerySQLCheckerTool(description='Use this tool to 

In [41]:
from langchain import hub

prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")

assert len(prompt_template.messages) == 1
prompt_template.messages[0].pretty_print()

================================ System Message ================================

You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.
You have access to tools for interacting with the database.
Only use the below tools. Only use the information returned by the below tools to construct your final answer.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to th

In [42]:
system_message = prompt_template.format(dialect="SQLite", top_k=5)

In [43]:
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, tools, state_modifier=system_message,
                                    )

In [44]:
db.run('''select * from  dataframe limit 10''')

"[('Cynthia Guerrero', 90, 'Port Eric'), ('Connie Chase', 66, 'West Courtney'), ('Jeremy Gonzalez', 74, 'Jeromemouth'), ('Thomas Hill', 36, 'Mooreview'), ('Kimberly Casey', 72, 'West Andrea'), ('Aaron Harmon', 64, 'West Natalieshire'), ('Matthew Ball', 31, 'North Cassandra'), ('Angelica Owens', 89, 'Jacksonberg'), ('Heather Roman', 21, 'Anthonyborough'), ('Brooke Robinson', 23, 'Lake Adamfort')]"

In [45]:
question = "Count the age of people whose age is 22"

for step in agent_executor.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Count the age of people whose age is 22
================================== Ai Message ==================================
Tool Calls:
  sql_db_query_checker (call_7mk4)
 Call ID: call_7mk4
  Args:
    query: SELECT * FROM people WHERE age = 22
================================= Tool Message =================================
Name: sql_db_query_checker

The query is correct and does not contain any common mistakes.
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (call_ny56)
 Call ID: call_ny56
  Args:
    query: SELECT COUNT(*) FROM people WHERE age = 22
================================= Tool Message =================================
Name: sql_db_query

Error: (sqlite3.OperationalError) no such table: people
[SQL: SELECT COUNT(*) FROM people WHERE age = 22]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
=================================